# Task 1.1 — Core Contribution / Architecture

**Paper:** *A Dual Coordinate Descent Method for Large-scale Linear SVM*  
**Authors:** Cho-Jui Hsieh, Kai-Wei Chang, Chih-Jen Lin, S. Sathiya Keerthi, S. Sundararajan  
**Venue:** ICML 2008  
**Student:** Navnit Naman | Roll: 230085

## Step-by-Step Method Description

### Step 1: Formulate the Dual Problem
- **Description:** The paper transforms the primal SVM problem (Eq. 1 in the paper) into its dual form. For both L1-SVM and L2-SVM, the dual is expressed as: minimize f(α) = ½ αᵀQ̄α − eᵀα, subject to 0 ≤ αᵢ ≤ U for all i (Eq. 4). Here Q̄ = Q + D, Qᵢⱼ = yᵢ yⱼ xᵢᵀ xⱼ, and D is a diagonal regularization matrix (zero for L1-SVM, 1/(2C) for L2-SVM).
- **Reference:** Equation (4), Section 1 (Introduction)
- **Purpose:** Casting the problem in the dual makes it possible to exploit the linear kernel structure: the gradient can be computed cheaply using w = Σ yᵢαᵢxᵢ (Eq. 12) without storing the full kernel matrix.

### Step 2: Initialize α and Compute w
- **Description:** Start from α⁰ = 0 (all dual variables at zero), which implies w = Σ yᵢαᵢxᵢ = 0. The vector w serves as a proxy for the full gradient of f, updated incrementally.
- **Reference:** Algorithm 1, beginning of Section 2
- **Purpose:** A zero initialization ensures feasibility (0 ≤ αᵢ ≤ U) from the start and provides a clean starting point.

### Step 3: Outer Iteration — Cycle Through All Variables
- **Description:** Each outer iteration k consists of l inner steps (one per training instance). The update order at each outer iteration is chosen **randomly** (a permutation of {1, …, l}) rather than a fixed cyclic order. This randomization empirically speeds convergence.
- **Reference:** Algorithm 1, Section 3.1 (Random Permutation)
- **Purpose:** Random permutation prevents adversarial orderings and ensures each instance is visited exactly once per outer iteration, balancing efficiency and convergence.

### Step 4: Inner Iteration — Solve a One-Variable Sub-Problem
- **Description:** For the selected index i, compute the partial gradient ∇ᵢ f(α) = Q̄ᵢᵢ αᵢ − 1 + yᵢ wᵀ xᵢ (using Eq. 10 and 12). Then solve the one-variable quadratic sub-problem (Eq. 6): minimize over d the function ½ Q̄ᵢᵢ d² + ∇ᵢ f(α) d, subject to 0 ≤ αᵢ + d ≤ U. The closed-form optimal update is: α_i_new = clip(αᵢ − ∇ᵢf / Q̄ᵢᵢ, 0, U).
- **Reference:** Equations (6)–(11), Section 2, Algorithm 1 steps (a)–(c)
- **Purpose:** This is the core coordinate descent step. Because the sub-problem has only one variable, it is solved exactly in O(n̄) time (where n̄ is the average number of nonzeros per instance), which is far cheaper than the O(l n̄) cost of gradient-maintaining decomposition methods.

### Step 5: Update the Weight Vector w
- **Description:** After updating αᵢ to αᵢ_new, update w ← w + (αᵢ_new − αᵢ) yᵢ xᵢ (Eq. 12). This O(n̄) update keeps w current so the gradient can be re-computed cheaply at the next inner step.
- **Reference:** Equation (12), Section 2
- **Purpose:** Avoids recomputing w from scratch at each step; this is the key efficiency advantage of linear SVM — no kernel matrix row is needed.

### Step 6: Optional Shrinking
- **Description:** Variables αᵢ that are provably at their bounds (0 or U) for many consecutive iterations can be temporarily removed from the active set A using conditions in Eq. (16), reducing the effective problem size. The shrunk variables are reconstructed when the stopping condition for the sub-problem on A is met.
- **Reference:** Section 3.2, Algorithm 3 (implied), Eq. (16)–(17)
- **Purpose:** Shrinking accelerates convergence by focusing computation on variables that are not yet at their optimal bounds.

### Step 7: Check Stopping Condition
- **Description:** At the end of each outer iteration, compute Mᵏ = max projected gradient and mᵏ = min projected gradient (Eq. 17). Stop when Mᵏ − mᵏ < ε, |Mᵏ| < ε, and |mᵏ| < ε. Theorem 1 guarantees this is satisfied in O(log(1/ε)) iterations (Section 2).
- **Reference:** Equation (17), Theorem 1, Section 2
- **Purpose:** The stopping criterion ensures convergence to an ε-optimal solution, and the O(log(1/ε)) bound makes the algorithm practical.

### Step 8: Recover Primal Solution
- **Description:** Once the dual optimal α∗ is found, the primal weight vector is directly available as w∗ = Σ yᵢαᵢ∗xᵢ (maintained throughout training). Prediction for a test point x is sign(wᵀx).
- **Reference:** Equation (12), Section 2
- **Purpose:** The primal solution w is obtained for free as a by-product of the dual optimization, requiring no additional computation.

---

## Final Summary Sentence

This paper solves the large-scale linear SVM training problem by applying coordinate descent directly to the dual formulation (rather than the primal), updating one dual variable αᵢ at a time using a cheap closed-form O(n̄) update that avoids storing the kernel matrix, and the authors claim this is faster than state-of-the-art solvers (Pegasos, TRON, SVMperf, primal coordinate descent) because it requires far fewer floating-point operations per iteration while still converging in O(log(1/ε)) outer iterations.